# PyVis — a network you can drag

**PyVis -- a network you can drag, for when the static picture is too dense.**

Past roughly fifty nodes a static network diagram becomes a hairball. An interactive one lets the reader pull nodes apart and follow a single thread, which is the one job static cannot do.

**What it shows:**

- turning a networkx graph into an interactive HTML page
- carrying real information in size, colour and hover text
- physics settings, which decide whether it settles or wobbles forever

In Streamlit, read the HTML and pass it to st.components.v1.html(...) -- the same trick fastapi/project/client.py uses for folium maps.

---

*Chapter:* `networks` — graphs, where position means nothing unless you say so  
*Run the cells in order.* Every figure is also written to `viz/output/networks/`, which is what the Streamlit gallery (`viz/project/gallery.py`) reads.


## Setup

These lines are how every notebook in the folder finds `vizkit.py`, which holds the save helpers and the seeded sample data. The data is seeded on purpose: your figures should come out identical to everyone else's.

`save_html()` writes each chart into `viz/output/` as a standalone page you can open, email or embed. The chart also renders below the cell, because the cell ends by naming it.


In [ ]:
# A notebook has no __file__, so find viz/ by walking up from this
# notebook's own folder until vizkit.py turns up.
import sys
from pathlib import Path

VIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
           if (p / "vizkit.py").exists())
sys.path.insert(0, str(VIZ))

import networkx as nx
from pyvis.network import Network

from vizkit import save_html, show_html

# Where save_html() files this lesson's output: viz/output/networks/
LESSON = "networks/pyvis_interactive"


## The graph, and what to encode

Same graph and the same two measurements as the static lesson — degree and community. The encodings do not change just because the output is interactive.


In [ ]:
graph = nx.karate_club_graph()
degrees = dict(graph.degree())
communities = nx.community.greedy_modularity_communities(graph)
community_of = {node: i for i, group in enumerate(communities) for node in group}
palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9"]


## Nodes and edges

`cdn_resources="in_line"` bundles vis.js into the file, so the saved page works offline and inside Streamlit. Node `title` is the hover text: the place to put detail the picture has no room for.


In [ ]:
network = Network(height="620px", width="100%", bgcolor="#ffffff",
                  font_color="#333333", notebook=False, cdn_resources="in_line")

for node in graph.nodes():
    network.add_node(
        node,
        label=str(node),
        # Same encodings as the static lesson: size = connections, colour = group.
        size=8 + degrees[node] * 1.6,
        color=palette[community_of[node] % len(palette)],
        # Hover text can hold what the picture has no room for.
        title=f"member {node}\nconnections: {degrees[node]}\n"
              f"community: {community_of[node]}",
    )

for source, target in graph.edges():
    network.add_edge(source, target, color="#DDDDDD")


## Physics, and writing the page

The physics settings decide whether the layout comes to rest. Too little damping and it never stops moving, which is genuinely unpleasant to read.


In [ ]:
# Physics decides whether the layout settles. Too little damping and it never
# stops moving, which is nauseating to read.
network.barnes_hut(gravity=-4000, central_gravity=0.3, spring_length=110,
                   spring_strength=0.02, damping=0.9)

path = save_html(network, LESSON, "karate-club")


### See it in the notebook

PyVis only knows how to write a file, so `show_html()` drops that file into an iframe. Drag a node and the physics re-settles around it.


In [ ]:
show_html(path)


## Rules of thumb

```text
Interactive networks are worth it above ~50 nodes, where a static picture
turns into a hairball. Below that, the static version prints and pastes.

Same encodings either way: size = a quantity, colour = a group,
hover = the detail there is no room for.
```


## Try it yourself

Edit the cells above and re-run them — that is what the notebook is for.

1. Set `damping=0.2` and re-run. How long does the graph take to settle, and would you show that to an audience?
2. Add the member's community and degree to the node `label` instead of the hover `title`. Which is more readable at 34 nodes? At 300?
3. Call `network.show_buttons(filter_=['physics'])` before saving and tune the layout in the browser. Copy the settings back into the code.


In [ ]:
# your turn


---

**Previous:** [`networks/networkx_basics`](networkx_basics.ipynb)  
**Next:** [`project/makeover`](../../project/makeover.ipynb)
